# Module 8.3 — CLI Tools & Automation

### Python Mastery · Track 8: Real-World Python

Command-line tools are how much automation actually runs: scripts that take arguments, do work, and report a result through an exit code. This module covers `argparse` from the standard library, an orientation to the friendlier `Click` library, configuration through environment variables, and exit codes for use in scripts and pipelines.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- A command-line tool is a `.py` file you run from a shell, so the examples write a script with `%%writefile` and run it with `!python`, passing arguments exactly as a user would. This shows the real behaviour, including help text and exit codes.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Parse positional and optional arguments with `argparse`.
2. Add types, defaults, choices, flags, and subcommands.
3. Read configuration from environment variables with sensible defaults.
4. Return meaningful exit codes and understand how shells use them.
5. Recognise what `Click` and `Typer` add over `argparse`.

**Prerequisites:** Tracks 1 to 6.

---

## Part 1 · `argparse` Basics: Positional and Optional Arguments

The standard `argparse` module turns command-line arguments into Python values and generates help text automatically. You create a parser, declare the arguments you expect, and call `parse_args()`. **Positional** arguments are required and identified by order; **optional** arguments start with `-` or `--` and usually have defaults. The cell writes a small tool and runs it with different arguments.

In [ ]:
%%writefile greet.py
import argparse

parser = argparse.ArgumentParser(description="Greet someone.")
parser.add_argument("name", help="who to greet")                 # positional, required
parser.add_argument("--times", type=int, default=1, help="how many times")   # optional

args = parser.parse_args()
for _ in range(args.times):
    print(f"Hello, {args.name}!")

In [ ]:
# Run it like a user would, passing a positional argument and an option.
!python greet.py Asha --times 3

In [ ]:
# argparse generates --help automatically from your declarations.
!python greet.py --help

## Part 2 · Types, Choices, Flags, and Validation

`argparse` can convert and validate arguments for you. `type=int` converts and rejects non-integers; `choices=[...]` restricts an argument to a fixed set; and `action="store_true"` makes a boolean **flag** that is `True` when present. If input is invalid, `argparse` prints an error and exits with a non-zero status, so bad usage fails loudly.

In [ ]:
%%writefile convert.py
import argparse

parser = argparse.ArgumentParser(description="Convert a temperature.")
parser.add_argument("value", type=float, help="the temperature to convert")
parser.add_argument("--to", choices=["c", "f"], required=True, help="target unit")
parser.add_argument("--verbose", action="store_true", help="show the formula")

args = parser.parse_args()

if args.to == "f":
    result = args.value * 9 / 5 + 32
    if args.verbose:
        print(f"formula: {args.value} * 9/5 + 32")
else:
    result = (args.value - 32) * 5 / 9
    if args.verbose:
        print(f"formula: ({args.value} - 32) * 5/9")

print(f"result: {result:.1f}")

In [ ]:
!python convert.py 100 --to f --verbose

In [ ]:
# An invalid choice fails with a clear message and a non-zero exit code.
!python convert.py 100 --to kelvin ; echo "exit code was: $?"

## Part 3 · Subcommands

Larger tools group functionality into **subcommands**, like `git commit` and `git push`. `argparse` supports this through subparsers: each subcommand has its own arguments, and the parser dispatches to the right one. This keeps a multi-purpose tool organised and discoverable.

In [ ]:
%%writefile todo.py
import argparse

parser = argparse.ArgumentParser(description="A tiny todo tool.")
subparsers = parser.add_subparsers(dest="command", required=True)

# Subcommand: add
add_parser = subparsers.add_parser("add", help="add a task")
add_parser.add_argument("task", help="the task text")

# Subcommand: list
list_parser = subparsers.add_parser("list", help="list tasks")
list_parser.add_argument("--all", action="store_true", help="include completed")

args = parser.parse_args()

if args.command == "add":
    print(f"added task: {args.task}")
elif args.command == "list":
    scope = "all" if args.all else "pending"
    print(f"listing {scope} tasks")

In [ ]:
!python todo.py add "write the notebook"
!python todo.py list --all

## Part 4 · Configuration and Exit Codes

Command-line tools often read **configuration** from environment variables, which lets behaviour change between machines without editing code (a practice from Module 6.6). And every program returns an **exit code**: `0` means success, and any non-zero value signals an error. Shells and automation rely on this, for example `command_a && command_b` only runs the second if the first returned `0`. Return codes with `sys.exit(code)`.

In [ ]:
%%writefile check.py
import argparse
import os
import sys

parser = argparse.ArgumentParser(description="Check a number against a threshold.")
parser.add_argument("number", type=int)
args = parser.parse_args()

# Read configuration from the environment, with a default.
threshold = int(os.environ.get("THRESHOLD", "10"))

if args.number >= threshold:
    print(f"{args.number} meets the threshold of {threshold}")
    sys.exit(0)                 # success
else:
    print(f"{args.number} is below the threshold of {threshold}")
    sys.exit(1)                 # failure, signalled to the shell

In [ ]:
# Default threshold is 10. 15 passes (exit 0); 5 fails (exit 1).
!python check.py 15 ; echo "exit: $?"
!python check.py 5  ; echo "exit: $?"

In [ ]:
# Override the configuration via an environment variable for one run.
!THRESHOLD=3 python check.py 5 ; echo "exit: $?"
# With THRESHOLD=3, the value 5 now passes.

## Part 5 · Beyond `argparse`: `Click` and `Typer`

`argparse` is capable and always available, but third-party libraries make complex CLIs more pleasant. **`Click`** uses decorators to define commands and options, handles a lot of boilerplate, and composes subcommands cleanly. **`Typer`** builds on Click and derives the interface from type hints, so a well-annotated function becomes a CLI almost for free. The cell shows a `Click` tool, which runs here; `Typer` is similar in spirit.

In [ ]:
%%writefile click_demo.py
import click

@click.command()
@click.argument("name")
@click.option("--count", default=1, help="number of greetings")
@click.option("--shout", is_flag=True, help="uppercase the greeting")
def greet(name, count, shout):
    """Greet NAME a number of times."""        # this docstring becomes the help text
    message = f"Hello, {name}!"
    if shout:
        message = message.upper()
    for _ in range(count):
        click.echo(message)

if __name__ == "__main__":
    greet()

In [ ]:
!python click_demo.py Asha --count 2 --shout

In [ ]:
# Click also generates help automatically, from the decorators and docstring.
!python click_demo.py --help

The `Click` version expresses the same tool with less plumbing than `argparse`, and `Typer` would let you write an ordinary type-annotated function and turn it into a CLI with almost no extra code. For simple scripts `argparse` is perfectly good and needs no dependency; for larger tools, `Click` or `Typer` improve ergonomics and maintainability.

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — One positional argument (Easy)

In [ ]:
%%writefile ex1.py
import argparse
p = argparse.ArgumentParser()
p.add_argument("word")
args = p.parse_args()
print(args.word.upper())

In [ ]:
!python ex1.py hello

### Example 2 — An optional with a default (Easy)

In [ ]:
%%writefile ex2.py
import argparse
p = argparse.ArgumentParser()
p.add_argument("--n", type=int, default=5)
args = p.parse_args()
print(list(range(args.n)))

In [ ]:
!python ex2.py
!python ex2.py --n 3

### Example 3 — A boolean flag (Medium)

In [ ]:
%%writefile ex3.py
import argparse
p = argparse.ArgumentParser()
p.add_argument("text")
p.add_argument("--reverse", action="store_true")
args = p.parse_args()
print(args.text[::-1] if args.reverse else args.text)

In [ ]:
!python ex3.py python --reverse
!python ex3.py python

### Example 4 — Choices with validation (Medium)

In [ ]:
%%writefile ex4.py
import argparse
p = argparse.ArgumentParser()
p.add_argument("--mode", choices=["fast", "slow"], required=True)
args = p.parse_args()
print(f"running in {args.mode} mode")

In [ ]:
!python ex4.py --mode fast
!python ex4.py --mode turbo ; echo "exit: $?"

### Example 5 — Subcommands dispatch (Difficult)

In [ ]:
%%writefile ex5.py
import argparse

parser = argparse.ArgumentParser()
subs = parser.add_subparsers(dest="cmd", required=True)

double = subs.add_parser("double")
double.add_argument("n", type=int)

repeat = subs.add_parser("repeat")
repeat.add_argument("text")
repeat.add_argument("--times", type=int, default=2)

args = parser.parse_args()
if args.cmd == "double":
    print(args.n * 2)
elif args.cmd == "repeat":
    print(args.text * args.times)

In [ ]:
!python ex5.py double 21
!python ex5.py repeat ab --times 3

### Example 6 — Exit codes in a pipeline (Difficult)

In [ ]:
%%writefile ex6.py
import argparse, sys

p = argparse.ArgumentParser()
p.add_argument("password")
args = p.parse_args()

# A simple validation tool: succeeds (exit 0) only if the password is strong enough.
strong = len(args.password) >= 8 and any(c.isdigit() for c in args.password)
print("strong" if strong else "weak")
sys.exit(0 if strong else 1)

In [ ]:
# The shell can chain on the exit code: only print OK if the check passed.
!python ex6.py abc123 ; echo "exit: $?"
!python ex6.py longpass9 && echo "accepted by the shell because exit was 0"

---

## Exercises

Write each tool with `%%writefile`, then run it with `!python`, passing arguments as shown above.

**Exercise 1 (Easy).** Write a CLI that takes one positional argument `name` and prints `"Hi, <name>"`. Run it with a name.

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !python run cell here


**Exercise 2 (Easy).** Write a CLI with an optional `--count` (int, default 1) that prints `"tick"` that many times. Run it with and without the option.

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !python run cell here


**Exercise 3 (Medium).** Write a CLI taking a positional `number` (int) and a `--squared` flag; print the number, or its square if the flag is set. Demonstrate both.

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !python run cell here


**Exercise 4 (Medium).** Write a CLI with `--unit` restricted to `choices=["kg", "lb"]` and a positional `value` (float); convert kg to lb (×2.205) or lb to kg (÷2.205) accordingly. Show a valid run and an invalid `--unit`.

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !python run cell here


**Exercise 5 (Difficult).** Write a CLI that reads an environment variable `LIMIT` (default 100), takes a positional `amount` (int), prints whether it is within the limit, and exits with code 0 if within and 1 if over. Demonstrate with the default and with an overridden `LIMIT`.

In [ ]:
# Your %%writefile cell here


In [ ]:
# Your !python run cell here


---

## Solutions

**Solution 1**

In [ ]:
%%writefile sol1.py
import argparse
p = argparse.ArgumentParser()
p.add_argument("name")
print(f"Hi, {p.parse_args().name}")

In [ ]:
!python sol1.py Asha

**Solution 2**

In [ ]:
%%writefile sol2.py
import argparse
p = argparse.ArgumentParser()
p.add_argument("--count", type=int, default=1)
for _ in range(p.parse_args().count):
    print("tick")

In [ ]:
!python sol2.py
!python sol2.py --count 3

**Solution 3**

In [ ]:
%%writefile sol3.py
import argparse
p = argparse.ArgumentParser()
p.add_argument("number", type=int)
p.add_argument("--squared", action="store_true")
args = p.parse_args()
print(args.number ** 2 if args.squared else args.number)

In [ ]:
!python sol3.py 5
!python sol3.py 5 --squared

**Solution 4**

In [ ]:
%%writefile sol4.py
import argparse
p = argparse.ArgumentParser()
p.add_argument("value", type=float)
p.add_argument("--unit", choices=["kg", "lb"], required=True)
args = p.parse_args()
if args.unit == "kg":
    print(f"{args.value * 2.205:.2f} lb")
else:
    print(f"{args.value / 2.205:.2f} kg")

In [ ]:
!python sol4.py 10 --unit kg
!python sol4.py 10 --unit stone ; echo "exit: $?"

**Solution 5**

In [ ]:
%%writefile sol5.py
import argparse, os, sys
p = argparse.ArgumentParser()
p.add_argument("amount", type=int)
args = p.parse_args()
limit = int(os.environ.get("LIMIT", "100"))
if args.amount <= limit:
    print(f"{args.amount} is within {limit}")
    sys.exit(0)
print(f"{args.amount} exceeds {limit}")
sys.exit(1)

In [ ]:
!python sol5.py 50 ; echo "exit: $?"
!LIMIT=20 python sol5.py 50 ; echo "exit: $?"

---

## Summary and Key Points

- `argparse` turns command-line arguments into Python values and auto-generates help: positional arguments are required and ordered, optional arguments (`--name`) usually have defaults.
- It converts and validates with `type=`, restricts with `choices=`, and defines boolean flags with `action="store_true"`; invalid input fails with a message and a non-zero exit.
- Subcommands via subparsers organise multi-purpose tools (like `git commit`), each with its own arguments.
- Tools read configuration from environment variables (with defaults) and return exit codes: `0` for success, non-zero for failure, which shells and pipelines depend on; set them with `sys.exit(code)`.
- `Click` (decorator-based) and `Typer` (type-hint-based) reduce boilerplate for larger CLIs; `argparse` remains ideal for simple scripts with no dependencies.

### Next module: 8.4 — Orientation: Web Frameworks

The next module orients you to web development in Python: the request/response model, WSGI versus ASGI, and when to reach for Flask (synchronous) versus FastAPI (asynchronous).